# Configuration

In [0]:
# Run-mode controls (F02.2).
#
# run_mode defaults to "test": every table this notebook touches is
# resolved to an isolated *_test Delta table through src.common.environment.
# "live" resolves the normal development tables and is used only after the
# test-mode evidence gate has passed.
dbutils.widgets.dropdown("run_mode", "test", ["test", "live"], "Run mode")
dbutils.widgets.dropdown("force_reload", "false", ["false", "true"], "Force reload")

In [0]:
# Clean and validate the Toronto Airbnb Bronze data into Silver.
#
# Silver keeps listings with a valid ID. Missing prices or coordinates are
# quality issues, not automatic rejection reasons.

from pyspark.sql import functions as F
from pyspark.sql.types import (
    DoubleType,
    IntegerType,
    StringType,
    StructField,
    StructType,
    TimestampType,
)
from pyspark.sql.window import Window

from pathlib import Path
import sys

start = Path.cwd().resolve()
ROOT = next(
    (p for p in [start, *start.parents]
     if (p / 'src').is_dir() and (p / 'config/project.yml').is_file()),
    None,
)
if ROOT is None:
    raise RuntimeError('Cannot locate repository root. Open this notebook inside the Databricks Git folder.')
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
print('Repository root:', ROOT)

from src.airbnb.transformations import clean_price
from src.common.environment import (
    resolve_tables,
    validate_force_reload,
    validate_run_mode,
)

CATALOG = "trippilot_dev"
BRONZE_SCHEMA = "bronze"
SILVER_SCHEMA = "silver"

CITY = "toronto"
SNAPSHOT_DATE = "2026-09-16"

# Run mode comes from the widget at the top of this notebook. Test mode
# resolves isolated *_test tables through src.common.environment, so a
# test run can never overwrite the development Silver tables.
#
# Silver rebuilds its output with overwrite semantics on every run, so it
# is already idempotent; force_reload is accepted for a consistent UI and
# does not change Silver behavior.
RUN_MODE = validate_run_mode(dbutils.widgets.get("run_mode"))
FORCE_RELOAD = validate_force_reload(dbutils.widgets.get("force_reload"))
TABLES = resolve_tables(RUN_MODE)

BRONZE_TABLE = TABLES["bronze"]
SILVER_TABLE = TABLES["silver"]
REJECTED_TABLE = TABLES["silver_rejected"]

print(f"Run mode: {RUN_MODE} (force_reload={FORCE_RELOAD})")
print(f"Bronze input: {BRONZE_TABLE}")
print(f"Silver output: {SILVER_TABLE}")
print(f"Rejected output: {REJECTED_TABLE}")

# Read only the Toronto Bronze slice

In [0]:
# Read only the Toronto snapshot from Bronze.
#
# Filtering by city and snapshot prevents Silver from accidentally
# processing data from another city or another source date.

bronze_df = (
    spark.table(BRONZE_TABLE)
    .filter(
        (F.col("_city") == CITY)
        & (F.col("_snapshot_date") == SNAPSHOT_DATE)
    )
)

bronze_row_count = bronze_df.count()

print(f"Bronze rows read: {bronze_row_count}")
display(bronze_df.limit(5))

# Select and rename business columns

In [0]:
# Select only the columns needed for the trip assistant and future metrics.
#
# Keep Bronze provenance columns so every Silver row remains traceable.

silver_base_df = bronze_df.select(
    F.col("id").alias("listing_id"),
    F.col("listing_url").alias("listing_url"),
    F.col("name").alias("listing_name"),
    F.col("host_id").alias("host_id"),
    F.col("host_name").alias("host_name"),
    F.col("neighbourhood_cleansed").alias("neighbourhood"),
    F.col("latitude").alias("latitude_raw"),
    F.col("longitude").alias("longitude_raw"),
    F.col("room_type").alias("room_type"),
    F.col("price").alias("price_raw"),
    F.col("minimum_nights").alias("minimum_nights_raw"),
    F.col("availability_365").alias("availability_365_raw"),
    F.col("last_review").alias("last_review_raw"),
    F.col("_source").alias("source"),
    F.col("_city").alias("city"),
    F.col("_snapshot_date").alias("snapshot_date"),
    F.col("_raw_path").alias("raw_path"),
    F.col("_run_id").alias("bronze_run_id"),
    F.col("_ingested_at_utc").alias("bronze_ingested_at_utc"),
)

display(silver_base_df.limit(5))

# Clean and cast fields

In [0]:
# Apply explicit Silver transformations.
#
# - Trim text fields.
# - Cast coordinates and numeric fields.
# - Use the reusable Python helper for price cleaning.
# - Keep invalid prices as null rather than guessing.

silver_df = (
    silver_base_df
    .withColumn(
        "listing_id",
        F.trim(F.col("listing_id")),
    )
    .withColumn(
        "listing_name",
        F.trim(F.col("listing_name")),
    )
    .withColumn(
        "neighbourhood",
        F.trim(F.col("neighbourhood")),
    )
    .withColumn(
        "room_type",
        F.trim(F.col("room_type")),
    )
    .withColumn(
        "latitude",
        F.col("latitude_raw").cast(DoubleType()),
    )
    .withColumn(
        "longitude",
        F.col("longitude_raw").cast(DoubleType()),
    )
    .withColumn(
        "minimum_nights",
        F.col("minimum_nights_raw").cast(IntegerType()),
    )
    .withColumn(
        "availability_365",
        F.col("availability_365_raw").cast(IntegerType()),
    )
    .withColumn(
        "price",
        F.udf(clean_price, DoubleType())(F.col("price_raw")),
    )
    .withColumn(
        "last_review",
        F.to_date(F.col("last_review_raw"), "yyyy-MM-dd"),
    )
)

display(silver_df.limit(5))

# Separate valid and invalid records

In [0]:
# Define the minimum requirement for a valid Silver listing.
#
# A listing must have a usable listing ID. Price and coordinates remain
# important, but missing values should not remove an otherwise useful
# listing from Silver.

invalid_id_condition = (
    F.col("listing_id").isNull()
    | (F.trim(F.col("listing_id")) == "")
)

valid_df = silver_df.filter(~invalid_id_condition)
rejected_df = silver_df.filter(invalid_id_condition)

# Add explicit quality flags for downstream Gold metrics and analytics.
valid_df = valid_df.withColumn(
    "has_valid_price",
    F.col("price").isNotNull(),
).withColumn(
    "has_valid_coordinates",
    F.col("latitude").isNotNull() & F.col("longitude").isNotNull(),
)

valid_row_count = valid_df.count()
rejected_row_count = rejected_df.count()

print(f"Valid Silver candidates: {valid_row_count}")
print(f"Rejected rows: {rejected_row_count}")
print(f"Reconciliation: {valid_row_count + rejected_row_count}")

display(rejected_df.limit(10))

# Remove Duplicate

In [0]:
# Remove duplicate listing IDs deterministically.
#
# Keep the most recently ingested Bronze record for each listing ID.
# Spark does not guarantee row order, so do not rely on dropDuplicates
# to preserve the "first" row.

dedup_window = Window.partitionBy("listing_id").orderBy(
    F.col("bronze_ingested_at_utc").desc(),
    F.col("bronze_run_id").desc(),
)

silver_deduplicated_df = (
    valid_df
    .withColumn("dedup_rank", F.row_number().over(dedup_window))
    .filter(F.col("dedup_rank") == 1)
    .drop("dedup_rank")
)

silver_row_count = silver_deduplicated_df.count()
duplicate_rows_removed = valid_row_count - silver_row_count

print(f"Silver rows after deduplication: {silver_row_count}")
print(f"Duplicate valid rows removed: {duplicate_rows_removed}")

# Write Silver and rejected-records tables

In [0]:
# Write valid Silver records.
#
# This one-city revision overwrites only the Toronto-derived Silver output.
# Registry-driven, multi-city-safe behavior arrives with a later roadmap
# feature; see docs/progress.md for the current roadmap position.

(
    silver_deduplicated_df
    .write
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(SILVER_TABLE)
)

# Write rejected records separately for investigation.
(
    rejected_df
    .write
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(REJECTED_TABLE)
)

print(f"Silver table written: {SILVER_TABLE}")
print(f"Rejected table written: {REJECTED_TABLE}")

# Verify Silver output

In [0]:
# Verify the revised Silver output.
#
# Checks:
# - No duplicate listing IDs
# - No null or blank listing IDs
# - Missing prices and coordinates are visible through quality flags
# - Bronze input reconciles to Silver, rejected, and duplicate rows

bronze_count = bronze_row_count
silver_count = spark.table(SILVER_TABLE).count()
rejected_count = spark.table(REJECTED_TABLE).count()

distinct_silver_ids = (
    spark.table(SILVER_TABLE)
    .select("listing_id")
    .distinct()
    .count()
)

verification_df = spark.table(SILVER_TABLE).agg(
    F.count("*").alias("silver_row_count"),
    F.countDistinct("listing_id").alias("distinct_listing_ids"),
    F.count(F.when(F.col("has_valid_price") == False, 1)).alias("missing_prices"),
    F.count(F.when(F.col("has_valid_coordinates") == False, 1)).alias("missing_coordinates"),
)

display(verification_df)

assert silver_count == distinct_silver_ids, (
    f"Duplicate listing IDs found: {silver_count} rows vs {distinct_silver_ids} IDs"
)

assert bronze_count == silver_count + rejected_count + duplicate_rows_removed, (
    f"Count mismatch: bronze={bronze_count}, silver={silver_count}, "
    f"rejected={rejected_count}, duplicates_removed={duplicate_rows_removed}"
)

print("Silver revision verification passed.")